# 06. Retriever

Retriever는 Vector Store를 **LangChain 체인에 연결 가능한 형태**로 감싼 인터페이스입니다.  
`as_retriever()`로 변환하면 LCEL 체인의 구성 요소로 사용할 수 있습니다.

In [1]:
import sys
sys.path.append("..")
from common_config import embedding_model

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 벡터 스토어 준비
loader = TextLoader("data/ai_basic.txt", encoding="utf-8")
docs = loader.load()

chunks = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50
).split_documents(docs)

# embedding 모델 객체생성
embeddings = embedding_model()

vector_store = FAISS.from_documents(chunks, embeddings)

print("준비 완료")

/tmp/ipykernel_42885/2427856007.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


준비 완료


## 1. 기본 Retriever 생성

In [3]:
# k: 반환할 문서 수
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

results = retriever.invoke("RAG란 무엇인가요?")

print(f"검색된 문서 수: {len(results)}")
for i, doc in enumerate(results):
    print(f"\n[{i+1}] {doc.page_content[:150]}")

검색된 문서 수: 3

[1] RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시

[2] 대규모 언어 모델(LLM)이란?
대규모 언어 모델(Large Language Model, LLM)은 방대한 텍스트 데이터로 학습된 트랜스포머 기반 모델입니다. GPT-4, Claude, Gemini 등이 대표적입니다. 텍스트 생성, 번역, 요약, 질의응답 등 다양한 언

[3] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다


## 2. 검색 타입 설정

| search_type | 설명 |
|---|---|
| `similarity` | 코사인 유사도 기반 (기본값) |
| `mmr` | 다양성을 고려한 검색 (중복 최소화), k 최종 반환할 문서 수, fetch_k MMR이 검토할 후보 문서 수  |
| `similarity_score_threshold` | 유사도 임계값 이상만 반환 |

In [4]:
# MMR - 유사하면서도 다양한 문서 검색
mmr_retriever = vector_store.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

results = mmr_retriever.invoke("RAG란 무엇인가요?")
for i, doc in enumerate(results):
    print(f"[{i+1}] {doc.page_content[:150]}")
    print("-"*30)

[1] RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시
------------------------------
[2] LangChain이란?
LangChain은 LLM 기반 애플리케이션 개발을 위한 오픈소스 프레임워크입니다. 모델, 프롬프트, 파서, 체인, 에이전트, 툴 등 다양한 컴포넌트를 제공합니다. Python과 JavaScript 버전이 있으며, RAG, 에이전트, 챗봇 등 다
------------------------------
[3] 합성곱 신경망(CNN, Convolutional Neural Network)은 이미지 처리에 특화된 신경망 구조입니다. 필터를 사용해 이미지의 특징을 추출하며, 이미지 분류, 객체 탐지에 사용됩니다.
순환 신경망(RNN, Recurrent Neural Network)은
------------------------------


In [5]:
# score_threshold - 유사도 임계값 설정
threshold_retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"score_threshold": 0.3, "k": 5}
)

results = threshold_retriever.invoke("RAG란 무엇인가요?")
print(f"임계값 이상 문서 수: {len(results)}")
for doc in results:
    print("-", doc.page_content[:100])

임계값 이상 문서 수: 1
- RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 


/home/aiuser/rag_agent_ex/.venv/lib/python3.12/site-packages/langchain_core/vectorstores/base.py:1048: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='876cf465-966c-4526-9b48-37c4a38412ed', metadata={'source': 'data/ai_basic.txt'}, page_content='RAG(Retrieval-Augmented Generation)이란?\nRAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.\nRAG 파이프라인은 문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어 저장 → 검색 → 답변 생성 순서로 진행됩니다.\n임베딩(Embedding)이란?'), np.float32(0.34987277)), (Document(id='0554eba4-62d7-4a86-bb6a-b876e7f93ad6', metadata={'source': 'data/ai_basic.txt'}, page_content='대규모 언어 모델(LLM)이란?\n대규모 언어 모델(Large Language Model, LLM)은 방대한 텍스트 데이터로 학습된 트랜스포머 기반 모델입니다. GPT-4, Claude, Gemini 등이 대표적입니다. 텍스트 생성, 번역, 요약, 질의응답 등 다양한 언어 작업을 수행할 수 있습니다.\nLLM의 한계로는 학습 데이터 이후의 최신 정보를 모른다는 점, 할루시네이션(Hallucination) 현상으로 사실과 다른 내용을 생성할 수 있다는 점이 있습니다.\nRAG(Retrieval-Augmented Generation)이란?'), np.float32(0.06955773)),

## 3. Retriever를 LCEL 체인에 연결

In [6]:
texts = ("aaa", "bbb", "cccc")
"\n\n".join(texts)

'aaa\n\nbbb\n\ncccc'

In [7]:
from langchain_core.runnables import RunnableLambda

retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x79e0b6c33170>, search_kwargs={'k': 3})

In [8]:
docs = retriever.invoke("임베딩이란?")
docs

[Document(id='9c39312e-6de2-4a06-a940-d65d5dfdecf6', metadata={'source': 'data/ai_basic.txt'}, page_content='임베딩(Embedding)이란?\n임베딩은 텍스트를 숫자 벡터로 변환하는 기술입니다. 의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다. OpenAI의 text-embedding-ada-002, text-embedding-3-small 등이 널리 사용됩니다.\n벡터 스토어(Vector Store)란?'),
 Document(id='876cf465-966c-4526-9b48-37c4a38412ed', metadata={'source': 'data/ai_basic.txt'}, page_content='RAG(Retrieval-Augmented Generation)이란?\nRAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.\nRAG 파이프라인은 문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어 저장 → 검색 → 답변 생성 순서로 진행됩니다.\n임베딩(Embedding)이란?'),
 Document(id='6a9ed1e9-4a24-41dc-a626-f4866de006f8', metadata={'source': 'data/ai_basic.txt'}, page_content='벡터 스토어(Vector Store)란?\n벡터 스토어는 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma, Pinecone, Weaviate 등이 대표적입니다. 코사인 유사도나 내적을 사용해 쿼리와 가장 유사한 문서를 빠르게 찾습니다.\nLangChain이란?')]

In [9]:
"\n\n".join([doc.page_content for doc in docs])

'임베딩(Embedding)이란?\n임베딩은 텍스트를 숫자 벡터로 변환하는 기술입니다. 의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다. OpenAI의 text-embedding-ada-002, text-embedding-3-small 등이 널리 사용됩니다.\n벡터 스토어(Vector Store)란?\n\nRAG(Retrieval-Augmented Generation)이란?\nRAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.\nRAG 파이프라인은 문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어 저장 → 검색 → 답변 생성 순서로 진행됩니다.\n임베딩(Embedding)이란?\n\n벡터 스토어(Vector Store)란?\n벡터 스토어는 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma, Pinecone, Weaviate 등이 대표적입니다. 코사인 유사도나 내적을 사용해 쿼리와 가장 유사한 문서를 빠르게 찾습니다.\nLangChain이란?'

In [10]:
# 검색 결과를 하나의 문자열로 합치는 함수
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# retriever → format 체인
retrieve_chain = retriever | RunnableLambda(format_docs)

context = retrieve_chain.invoke("임베딩이란?")
print(type(context))
print(context)

<class 'str'>
임베딩(Embedding)이란?
임베딩은 텍스트를 숫자 벡터로 변환하는 기술입니다. 의미가 비슷한 텍스트는 벡터 공간에서 가까운 위치에 놓입니다. OpenAI의 text-embedding-ada-002, text-embedding-3-small 등이 널리 사용됩니다.
벡터 스토어(Vector Store)란?

RAG(Retrieval-Augmented Generation)이란?
RAG는 검색(Retrieval)과 생성(Generation)을 결합한 기술입니다. LLM의 한계를 보완하기 위해 외부 문서를 검색하여 그 내용을 바탕으로 답변을 생성합니다. 최신 정보 제공, 할루시네이션 감소, 출처 제공이 가능합니다.
RAG 파이프라인은 문서 로딩 → 텍스트 분할 → 임베딩 → 벡터 스토어 저장 → 검색 → 답변 생성 순서로 진행됩니다.
임베딩(Embedding)이란?

벡터 스토어(Vector Store)란?
벡터 스토어는 임베딩 벡터를 저장하고 유사도 검색을 수행하는 데이터베이스입니다. FAISS, Chroma, Pinecone, Weaviate 등이 대표적입니다. 코사인 유사도나 내적을 사용해 쿼리와 가장 유사한 문서를 빠르게 찾습니다.
LangChain이란?


## 4. MultiQueryRetriever 맛보기

- 단일 쿼리를 여러 쿼리로 확장해 더 풍부한 검색 결과를 가져옴(Advanced RAG에서 자세히 다룹니다)  
- 하나의 입력 질문에 LLM이 여러 관점의 검색 질문을 바꿔서 검색하고 결과를 가져옴

- langchain.retrievers 모듈을 쓰려면 langchain-classic을 설치해야함.  
    ```uv add langchain-classic```

In [14]:
from langchain_classic.retrievers import MultiQueryRetriever
from common_config import llm_connect

llm = llm_connect()

base_retriever = vector_store.as_retriever(search_kwargs={"k": 3})

base_retriever
# llm이 사용자 질문에 대해서 여러 관점에서 검색을 수행하도록 객체를 생성함
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=base_retriever,
    llm=llm, 
    # include_original=True # 원래 질문을 검색에 포함시킬 경우, 기본은 False
)

# # llm이 여러 관점에서 질문을 만들어서, 벡터 DB 유사도 검색 수행,  문서를 검색하고, 각
results = multi_retriever.invoke("AI의 학습 방법을 알려줘")

results 
print(f"검색된 문서 수: {len(results)}")
for doc in results[:3]:
    print("-", doc.page_content[:120])

검색된 문서 수: 4
- 인공지능(AI)의 기초
인공지능이란 무엇인가?
인공지능(Artificial Intelligence, AI)은 컴퓨터 시스템이 인간의 지능적 행동을 모방하도록 하는 기술입니다. 학습, 추론, 문제 해결, 언어 이해 등
- 강화학습(Reinforcement Learning)은 에이전트가 환경과 상호작용하면서 보상을 최대화하는 방향으로 학습합니다. 게임 AI, 로봇 제어 등에 활용됩니다.
딥러닝이란?
딥러닝(Deep Learning)은 
- 지도학습(Supervised Learning)은 입력과 정답 레이블이 쌍으로 주어진 데이터를 학습합니다. 분류(Classification)와 회귀(Regression) 문제에 사용됩니다. 대표적인 알고리즘으로는 선형


## 정리

- `as_retriever()`로 VectorStore를 LCEL 체인에 연결합니다.
- `similarity`, `mmr`, `similarity_score_threshold` 검색 방식 선택 가능
- Retriever는 `invoke(query)` → `List[Document]` 반환

다음 단계: **전체 RAG 파이프라인 구현** →